
# Quick Start Exam Workflows

This notebook is intentionally self-contained. It does not import any local utility module, so you can open this file alone during study and see the full statistical workflow, simulation setup, model fitting, evaluation, plotting, and interpretation pattern in one place.



## What problem are we solving?

Most exam problems reduce to four decisions: identify whether the response is numeric or categorical, choose a reasonable model family, choose the correct metric, and evaluate performance without training on the test data.

**Highest-probability exam pattern:** Given a short prompt, write a complete simulate -> split/CV -> fit -> predict -> evaluate -> interpret workflow.



## Assumptions, intuition, and theory

Prediction error must be estimated on held-out data or by cross-validation. MSE/RMSE answer numeric prediction questions; accuracy, confusion matrices, sensitivity, and specificity answer classification questions. A simulation answer should report both the central result and variability across repetitions.



    ## Python method notes and documentation

    `fit` learns model parameters from training data, `predict` produces responses or labels, `mean_squared_error` measures numeric prediction error, `accuracy_score` measures class-label agreement, and `train_test_split` creates an honest holdout set.

    Documentation links:

    - [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)
- [train_test_split](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html)
- [mean_squared_error](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html)
- [accuracy_score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html)
- [KFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.KFold.html)
- [StratifiedKFold](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html)
- [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html)
- [LeaveOneOut](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneOut.html)
- [Pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html)
- [make_pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.make_pipeline.html)
- [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)
- [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)
- [KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html)
- [make_classification](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_classification.html)
- [make_moons](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_moons.html)
- [make_blobs](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.make_blobs.html)


## Fully self-contained worked example

Every code line below is commented so the workflow can be scanned quickly under exam time pressure.

In [ ]:
import numpy as np  # Import numerical arrays and random-number tools.
import pandas as pd  # Import DataFrame tables for compact summaries.
import matplotlib.pyplot as plt  # Import plotting tools for exam-ready figures.
from sklearn.base import clone  # Import clone so each model fit starts fresh.
from sklearn.datasets import make_classification, make_moons  # Import simulation helpers for labeled examples.
from sklearn.linear_model import LinearRegression, LogisticRegression  # Import baseline regression and classification models.
from sklearn.metrics import accuracy_score, mean_squared_error, confusion_matrix  # Import explicit performance metrics.
from sklearn.model_selection import train_test_split, KFold, cross_val_score  # Import train/test and CV tools.
from sklearn.neighbors import KNeighborsClassifier  # Import KNN for a flexible classifier example.
from sklearn.pipeline import make_pipeline  # Import pipeline to chain preprocessing and models.
from sklearn.preprocessing import PolynomialFeatures  # Import polynomial basis expansion.
RANDOM_SEED = 123  # Store the random seed in one visible place.
np.random.seed(RANDOM_SEED)  # Fix legacy NumPy randomness used by some libraries.
plt.style.use('default')  # Use a predictable matplotlib style.


In [ ]:
def make_sine_regression_data(n=120, noise=0.35, random_state=123):  # Define a reusable nonlinear regression simulator.
    rng = np.random.default_rng(random_state)  # Create a reproducible random number generator.
    x = np.sort(rng.uniform(0.0, 3.0, size=n))  # Draw predictor values and sort them for cleaner plots.
    signal = 4.0 + np.sin(3.0 * x)  # Define the true nonlinear mean function.
    y = signal + rng.normal(0.0, noise, size=n)  # Add Gaussian noise to create observed responses.
    X = x.reshape(-1, 1)  # Convert the predictor to a two-dimensional sklearn design matrix.
    return X, y, signal  # Return predictors, observed response, and noiseless truth.

def make_polynomial_regression_data(n=140, noise=2.0, random_state=123):  # Define a curved polynomial regression simulator.
    rng = np.random.default_rng(random_state)  # Create a reproducible random number generator.
    x = rng.uniform(-2.5, 2.5, size=n)  # Draw one numeric predictor over a moderate range.
    signal = 1.0 - 2.0 * x + 0.8 * x**2 + 0.7 * x**3  # Define the true polynomial mean curve.
    y = signal + rng.normal(0.0, noise, size=n)  # Add independent Gaussian errors.
    X = x.reshape(-1, 1)  # Reshape the predictor for sklearn estimators.
    return X, y, signal  # Return the design matrix, response, and true mean.

def train_test_mse(model, X, y, test_size=0.30, random_state=123):  # Define a local train/test regression evaluator.
    X_train, X_test, y_train, y_test = train_test_split(  # Split observations into training and testing parts.
        X,  # Pass the predictor matrix to the splitter.
        y,  # Pass the numeric response vector to the splitter.
        test_size=test_size,  # Hold out this fraction for honest testing.
        random_state=random_state,  # Fix the random split for reproducibility.
    )  # Finish the train/test split call.
    fitted_model = clone(model)  # Clone the estimator so repeated calls do not reuse fitted state.
    fitted_model.fit(X_train, y_train)  # Fit the model only on the training data.
    train_pred = fitted_model.predict(X_train)  # Predict the training responses to assess in-sample error.
    test_pred = fitted_model.predict(X_test)  # Predict the held-out responses to assess future error.
    train_mse = mean_squared_error(y_train, train_pred)  # Compute training mean squared error.
    test_mse = mean_squared_error(y_test, test_pred)  # Compute test mean squared error.
    return fitted_model, train_mse, test_mse  # Return the fitted model and both error estimates.


def make_gaussian_classification_data(n=180, class_sep=1.3, random_state=123):  # Define a reusable two-class Gaussian-like simulator.
    X, y = make_classification(  # Ask sklearn to create a labeled binary classification data set.
        n_samples=n,  # Set the number of observations.
        n_features=2,  # Keep two predictors so boundaries can be plotted.
        n_informative=2,  # Make both predictors useful for classification.
        n_redundant=0,  # Avoid redundant columns that distract from the exam idea.
        n_clusters_per_class=1,  # Use one cluster per class for a clean LDA-style example.
        class_sep=class_sep,  # Control how strongly separated the classes are.
        random_state=random_state,  # Fix the data simulation.
    )  # Finish the sklearn simulator call.
    return X, y  # Return predictors and class labels.

def make_moon_classification_data(n=200, noise=0.25, random_state=123):  # Define a nonlinear two-class simulator.
    X, y = make_moons(  # Generate two interlocking moon-shaped classes.
        n_samples=n,  # Set the number of observations.
        noise=noise,  # Add noise so the classification task is realistic.
        random_state=random_state,  # Fix the simulated data set.
    )  # Finish the make_moons call.
    return X, y  # Return predictors and labels.

def train_test_accuracy(model, X, y, test_size=0.30, random_state=123):  # Define a local train/test classification evaluator.
    X_train, X_test, y_train, y_test = train_test_split(  # Split data before fitting the classifier.
        X,  # Pass predictor matrix to the splitter.
        y,  # Pass class labels to the splitter.
        test_size=test_size,  # Hold out this fraction for testing.
        random_state=random_state,  # Make the split reproducible.
        stratify=y,  # Preserve the class proportions in train and test sets.
    )  # Finish the split call.
    fitted_model = clone(model)  # Clone the model to avoid carrying over old fitted state.
    fitted_model.fit(X_train, y_train)  # Fit the classifier using training observations only.
    train_pred = fitted_model.predict(X_train)  # Predict the training labels.
    test_pred = fitted_model.predict(X_test)  # Predict the held-out test labels.
    train_acc = accuracy_score(y_train, train_pred)  # Compute training accuracy.
    test_acc = accuracy_score(y_test, test_pred)  # Compute test accuracy.
    return fitted_model, train_acc, test_acc, y_test, test_pred  # Return model, accuracies, and test predictions.


In [ ]:
X_reg, y_reg, true_signal = make_sine_regression_data(n=120, noise=0.35, random_state=RANDOM_SEED)  # Simulate a numeric-response exam problem.
regression_model = LinearRegression()  # Choose a simple baseline regression method.
fitted_regression, train_mse, test_mse = train_test_mse(regression_model, X_reg, y_reg, random_state=RANDOM_SEED)  # Fit and evaluate by train/test split.
X_clf, y_clf = make_gaussian_classification_data(n=160, class_sep=1.4, random_state=RANDOM_SEED)  # Simulate a categorical-response exam problem.
classification_model = KNeighborsClassifier(n_neighbors=7)  # Choose a flexible classifier with a visible tuning parameter.
fitted_classifier, train_acc, test_acc, y_test, test_pred = train_test_accuracy(classification_model, X_clf, y_clf, random_state=RANDOM_SEED)  # Fit and evaluate classification accuracy.
summary = pd.DataFrame(  # Build one compact table that separates response types and metrics.
    [  # Start the list of table rows.
        {'problem_type': 'numeric prediction', 'method': 'linear regression', 'train_metric': train_mse, 'test_metric': test_mse, 'metric_name': 'MSE'},  # Add the regression result row.
        {'problem_type': 'class prediction', 'method': 'KNN, k=7', 'train_metric': train_acc, 'test_metric': test_acc, 'metric_name': 'accuracy'},  # Add the classification result row.
    ]  # End the row list.
)  # Finish the DataFrame construction.
display(summary)  # Display the workflow summary table.
confusion = confusion_matrix(y_test, test_pred)  # Compute the classification confusion matrix.
print('Confusion matrix for the held-out classification data:')  # Label the matrix output.
print(confusion)  # Print the confusion matrix.
plt.figure(figsize=(6, 4))  # Create a compact regression figure.
plt.scatter(X_reg[:, 0], y_reg, s=22, alpha=0.65, label='observed data')  # Plot the observed regression data.
plt.plot(X_reg[:, 0], fitted_regression.predict(X_reg), color='red', label='linear fit')  # Overlay the fitted regression line.
plt.xlabel('x')  # Label the horizontal axis.
plt.ylabel('y')  # Label the vertical axis.
plt.title('Quick regression workflow')  # Title the plot with the workflow purpose.
plt.legend()  # Show plot labels.
plt.show()  # Render the plot.



## Exam-style problem prompt

A prompt gives either a numeric response or a categorical response. Build a minimal but complete analysis that evaluates future prediction performance and produces one table or plot plus one sentence of interpretation.



    ## Adaptable code pattern

    ```python
    # Step 1: Decide whether y is numeric or categorical.
# Step 2: Simulate or load X and y.
# Step 3: Split into train/test data or choose a cross-validation splitter.
# Step 4: Fit the model only on training folds.
# Step 5: Predict held-out observations.
# Step 6: Compute MSE/RMSE for regression or accuracy/confusion matrix for classification.
# Step 7: Summarize the result in a small table or plot and interpret briefly.
    ```



## What to remember for the exam

Do not start by coding a fancy model. Start by naming the response type, metric, and validation strategy. Then use a small reproducible workflow that separates fitting data from evaluation data.
